A Camada Gold (A Regra de Negócio)
    
Agora você vai preparar o dado mastigado para a equipe de BI ou para a Diretoria da ANAC. Leia os dados limpos da Silver e atenda ao pedido do negócio.

A Ponte: Transforme o seu DataFrame da Silver em uma Temp View (tabela temporária) para podermos usar SQL.

A Query de Negócio: Usando Spark SQL, crie uma tabela consolidada que responda:

Qual é o total de ocorrências e o total de lesões fatais de passageiros agrupado por Operador (GOL, LATAM, AZUL, etc.) e por Estado (UF)?

Filtre a query para trazer apenas ocorrências no estado de São Paulo ('SP') ou Rio de Janeiro ('RJ').

Particionamento Físico: Pegue o resultado dessa query e salve no diretório /Volumes/.../Gold/anac_relatorio_operadores.

Requisito Técnico Final: Na hora de salvar, crie as sub-pastas organizadas por estado.

In [0]:
# transformar os dados da silver em uma tabela temp na gold
spark.sql("""CREATE OR REPLACE TEMPORARY VIEW vw_silver_anac
                            USING parquet
                            options (path = '/Volumes/pratica/pratica_dbs/silver_anac/anac_cleaned/') """)

In [0]:
%sql
-- tabela temp para fazer a tratativa dos requisitos tecnicos.
SELECT
  Operador_Padronizado,
  UF,
  COUNT(Numero_da_Ocorrencia) as TOTAL_DE_OCORRENCIAS,
  SUM(Lesoes_Fatais_Passageiros) AS LESOES_FATAIS_TOTAIS
FROM
  vw_silver_anac
WHERE
  UF = 'SP'
  OR UF = 'RJ'
GROUP BY
  Operador_Padronizado,
  UF
  LIMIT 5;

In [0]:
%sql
-- Criando a tabela fisica com base no que foi feito na tabela temp.
CREATE OR REPLACE TABLE pratica.pratica_dbs.table_gold AS
SELECT
  Operador_Padronizado,
  UF,
  COUNT(Numero_da_Ocorrencia) as TOTAL_DE_OCORRENCIAS,
  SUM(Lesoes_Fatais_Passageiros) AS LESOES_FATAIS_TOTAIS
FROM
  vw_silver_anac
WHERE
  UF = 'SP'
  OR UF = 'RJ'
GROUP BY
  Operador_Padronizado,
  UF;

### O resultado da query foi salvo na var (df_path_gold_volume).

### O save foi feito usando partitionBy() para salvar os valores por esatdo.

In [0]:
df_gold_path_table = spark.table("pratica.pratica_dbs.table_gold")

# caminho onde a tabela gold sera salva
df_path_gold_volume = "/Volumes/pratica/pratica_dbs/gold_anac/anac_relatorio_operadores"

(
    df_gold_path_table.write.format("parquet")
    .option("compression", "snappy")
    .partitionBy("UF")
    .mode("overwrite")
    .save(df_path_gold_volume)
)